In [13]:
# Cargar los datos
df_original = pd.read_csv("DataSet_House_Price_Prediction.csv")

# Hacer una copia
df = df_original.copy()

# Fijar la semilla
np.random.seed(42)

# Seleccionar el 1% de los datos
n_missing = int(len(df) * 0.01)

# Escoger las filas al azar
indices_missing = np.random.choice(
    df.index,
    size=n_missing,
    replace=False
)

# Agregar los datos faltantes
df.loc[indices_missing, "sqft_living"] = np.nan

# Revisar los valores faltantes
print(df.isnull().sum())

# Revisar la información general de los datos
print("Dimensiones:", df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nResumen de las variables numéricas:")
display(df.describe())

# Histogramas de las variables principales
import matplotlib.pyplot as plt

df[["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]].hist(
    figsize=(12, 8),
    bins=30
)

plt.tight_layout()
plt.show()

# Revisar posibles valores atípicos
columnas = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]

for columna in columnas:
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    cantidad = ((df[columna] < limite_inferior) |
                (df[columna] > limite_superior)).sum()

    print(columna, ":", cantidad)

    # Diagramas de caja para revisar valores atípicos
import matplotlib.pyplot as plt

columnas = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]

for columna in columnas:
    plt.figure(figsize=(8, 2.5))
    plt.boxplot(df[columna].dropna(), vert=False)
    plt.title(columna)
    plt.xlabel(columna)
    plt.show()

    # Revisar los valores extremos de las variables principales

columnas = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]

for columna in columnas:
    print("\n", columna.upper())
    print(df[columna].sort_values(ascending=False).head(10))

    # Revisar las viviendas con precios más altos
df.sort_values("price", ascending=False).head(10)

# Revisar viviendas con precio igual a cero
precio_cero = df[df["price"] == 0]

print("Cantidad de viviendas con precio 0:", len(precio_cero))
display(precio_cero)

# Eliminar registros con precio igual a cero
df = df[df["price"] > 0].copy()

print("Número de registros después de eliminar precios en cero:", len(df))



FileNotFoundError: [Errno 2] No such file or directory: 'DataSet_House_Price_Prediction.csv'

In [ ]:
print("Registros actuales:", len(df))
print("Precios en cero:", (df["price"] == 0).sum())

In [ ]:
# Revisar el registro con el precio más alto
precio_maximo = df.loc[df["price"].idxmax()]

print("Precio:", precio_maximo["price"])
print("Área habitable:", precio_maximo["sqft_living"])
print("Precio por pie cuadrado:", precio_maximo["price_per_sqft"])

# Comparar con el resto de viviendas
print("\nMediana del precio por pie cuadrado:",
      df["price_per_sqft"].median())

In [ ]:
# Revisar los precios por pie cuadrado más altos
df[["price", "sqft_living", "price_per_sqft"]].sort_values(
    "price_per_sqft", ascending=False
).head(15)

In [ ]:
# Revisar los cinco casos con mayor precio por pie cuadrado
df.loc[
    df["price_per_sqft"].nlargest(5).index,
    ["price", "sqft_living", "sqft_lot", "bedrooms",
     "bathrooms", "waterfront", "view", "condition",
     "street", "city", "statezip", "price_per_sqft"]
]

In [6]:
print("Dimensiones actuales:", df.shape)
print("Precios en cero:", (df["price"] == 0).sum())
print("Nulos en sqft_living:", df["sqft_living"].isnull().sum())


NameError: name 'df' is not defined

In [11]:
import pandas as pd
import numpy as np

In [14]:
import pandas as pd
import numpy as np

df_original = pd.read_csv("DataSet_House_Price_Prediction.csv")
df = df_original.copy()

print(df.shape)

(4600, 18)


In [15]:
# Agregar aproximadamente 1% de valores faltantes en sqft_living
np.random.seed(42)

cantidad_nulos = int(len(df) * 0.01)
filas_nulas = np.random.choice(df.index, cantidad_nulos, replace=False)

df.loc[filas_nulas, "sqft_living"] = np.nan

print("Nulos en sqft_living:", df["sqft_living"].isnull().sum())

Nulos en sqft_living: 46


In [16]:
# Eliminar registros con precio igual a cero
df = df[df["price"] > 0].copy()

print("Dimensiones actuales:", df.shape)
print("Precios en cero:", (df["price"] == 0).sum())
print("Nulos en sqft_living:", df["sqft_living"].isnull().sum())

Dimensiones actuales: (4551, 18)
Precios en cero: 0
Nulos en sqft_living: 45


In [17]:
# Calcular la mediana de sqft_living
mediana_sqft = df["sqft_living"].median()

# Reemplazar los valores faltantes por la mediana
df["sqft_living"] = df["sqft_living"].fillna(mediana_sqft)

print("Mediana utilizada:", mediana_sqft)
print("Nulos restantes:", df["sqft_living"].isnull().sum())
print("Dimensiones:", df.shape)

Mediana utilizada: 1970.0
Nulos restantes: 0
Dimensiones: (4551, 18)


In [19]:
# Variable objetivo
y = df["price"]

# Variables predictoras
X = df.drop(columns=["price", "price_per_sqft"])

print("Variables predictoras:")
print(X.columns.tolist())

print("\nCantidad de variables predictoras:", X.shape[1])
print("Variable objetivo: price")

Variables predictoras:
['date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city', 'statezip']

Cantidad de variables predictoras: 16
Variable objetivo: price


In [20]:
# Convertir date a formato de fecha
X["date"] = pd.to_datetime(X["date"])

# Extraer año y mes
X["year_sale"] = X["date"].dt.year
X["month_sale"] = X["date"].dt.month

# Eliminar la fecha original
X = X.drop(columns=["date"])

print("Cantidad de variables:", X.shape[1])
print(X.dtypes)

Cantidad de variables: 17
bedrooms         float64
bathrooms        float64
sqft_living      float64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street            object
city              object
statezip          object
year_sale          int32
month_sale         int32
dtype: object


In [21]:
# Revisar cantidad de categorías en las variables de ubicación
for columna in ["street", "city", "statezip"]:
    print(columna, ":", X[columna].nunique(), "categorías")

street : 4476 categorías
city : 44 categorías
statezip : 77 categorías


In [22]:
# Eliminar street por su alta cardinalidad
X = X.drop(columns=["street"])

print("Dimensiones de X:", X.shape)
print("Variables restantes:")
print(X.columns.tolist())

Dimensiones de X: (4551, 16)
Variables restantes:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'city', 'statezip', 'year_sale', 'month_sale']


In [23]:
from sklearn.model_selection import train_test_split

# Dividir los datos: 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (3640, 16)
Prueba: (911, 16)


In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Variables categóricas
categoricas = ["city", "statezip"]

# Variables numéricas
numericas = [col for col in X_train.columns if col not in categoricas]

# Preprocesamiento
preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", "passthrough", numericas)
    ]
)

print("Variables categóricas:", categoricas)
print("Cantidad de variables numéricas:", len(numericas))

Variables categóricas: ['city', 'statezip']
Cantidad de variables numéricas: 14


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Crear el modelo
modelo_lineal = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", LinearRegression())
])

# Entrenar el modelo
modelo_lineal.fit(X_train, y_train)

# Realizar predicciones
y_pred = modelo_lineal.predict(X_test)

# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 4))

MAE: 152702.62
RMSE: 243341.29
R²: 0.602


In [26]:
from sklearn.ensemble import RandomForestRegressor

# Crear modelo Random Forest
modelo_rf = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Entrenar
modelo_rf.fit(X_train, y_train)

# Predicciones
y_pred_rf = modelo_rf.predict(X_test)

# Métricas
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))
print("R²:", round(r2_rf, 4))

MAE: 133869.33
RMSE: 534737.36
R²: -0.922


In [27]:
from sklearn.ensemble import GradientBoostingRegressor

# Crear modelo Gradient Boosting
modelo_gb = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# Entrenar el modelo
modelo_gb.fit(X_train, y_train)

# Realizar predicciones
y_pred_gb = modelo_gb.predict(X_test)

# Calcular métricas
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("MAE:", round(mae_gb, 2))
print("RMSE:", round(rmse_gb, 2))
print("R²:", round(r2_gb, 4))

MAE: 129694.3
RMSE: 247623.76
R²: 0.5878


In [28]:
resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Random Forest", "Gradient Boosting"],
    "MAE": [mae, mae_rf, mae_gb],
    "RMSE": [rmse, rmse_rf, rmse_gb],
    "R2": [r2, r2_rf, r2_gb]
})

resultados.round(2)

,Modelo,MAE,RMSE,R2
0,Regresión Lineal,152702.62,243341.29,0.60
1,Random Forest,133869.33,534737.36,-0.92
2,Gradient Boosting,129694.30,247623.76,0.59


In [29]:
resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Random Forest", "Gradient Boosting"],
    "MAE": [mae, mae_rf, mae_gb],
    "RMSE": [rmse, rmse_rf, rmse_gb],
    "R2": [r2, r2_rf, r2_gb]
})

resultados.round(2)

,Modelo,MAE,RMSE,R2
0,Regresión Lineal,152702.62,243341.29,0.60
1,Random Forest,133869.33,534737.36,-0.92
2,Gradient Boosting,129694.30,247623.76,0.59
